# Silica Concentration 예측 — Mining Process Flotation Plant

**목표**: 광물 선광(flotation) 공정 데이터로 최종 정광의 **% Silica Concentrate** 를 예측합니다.

**데이터**: `dataset/day2_miniproject_reg.csv` (2017.03 ~ 2017.09, 1시간 간격)

| 변수 | 설명 |
|:---:|:---|
| date | 측정 일시 *(회귀 제외)* |
| % Iron Feed, % Silica Feed | 투입 원광의 철/실리카 품위 |
| Starch Flow, Amina Flow | 전분/아민 시약 투입량 |
| Ore Pulp Flow, pH, Density | 광액 유량/산도/밀도 |
| Flotation Column 01~07 Air Flow / Level | 컬럼별 공기 유량·액위 |
| % Iron Concentrate | 최종 정광 철 비율 (실험실 측정) |
| **% Silica Concentrate** | **회귀 타겟 — 최종 정광 실리카(불순물) 비율** |

**분석 흐름**
1. 데이터 탐색 및 전처리
2. `date` 컬럼 시계열 분석 → 회귀 포함 여부 결정
3. 다중선형회귀 / Ridge / Lasso / ElasticNet 학습 및 alpha 최적화
4. 모델 성능·회귀계수 비교

## 0) 분석 준비

In [ ]:
import os
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import rcParams

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, make_scorer
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.graphics.tsaplots import plot_acf

# ===== 한글 폰트 설정 =====
FONT_CANDIDATES = [
    "/System/Library/Fonts/AppleSDGothicNeo.ttc",
    "/System/Library/Fonts/Supplemental/AppleGothic.ttf",
    "C:/Windows/Fonts/malgun.ttf",
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
]
FONT_PATH = next((p for p in FONT_CANDIDATES if os.path.exists(p)), None)
if FONT_PATH:
    fm.fontManager.addfont(FONT_PATH)
    KOREAN_FONT = fm.FontProperties(fname=FONT_PATH)
    rcParams["font.family"] = KOREAN_FONT.get_name()
else:
    KOREAN_FONT = None

rcParams["axes.unicode_minus"] = False

TARGET = "% Silica Concentrate"
EXCLUDE_COLS = ["date"]  # 시계열 분석 후 제외
RANDOM_STATE = 0
TEST_SIZE = 0.3

print("한글 폰트:", KOREAN_FONT.get_name() if KOREAN_FONT else "없음")

## 1) 데이터 불러오기

In [ ]:
DATA_PATH = os.path.join(os.getcwd(), "dataset", "day2_miniproject_reg.csv")
df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df.head()

---## 2) 데이터 탐색 및 전처리### 2-1) 구조적·통계적 정보

In [ ]:
print("컬럼:", df.columns.tolist())
print("shape:", df.shape)
df.info()

In [ ]:
df.describe().T

In [ ]:
# 결측치 확인
df.isnull().sum()

### 2-2) 상관행렬 히트맵 (다중공선성 확인)

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(14, 11))
sns.heatmap(
    corr,
    vmin=-1, vmax=1, center=0,
    cmap=sns.diverging_palette(20, 220, n=200),
    annot=False,
    square=True,
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

# 타겟과 상관이 높은 변수
corr[TARGET].sort_values(key=abs, ascending=False).head(10)

> Flotation 컬럼별 Air Flow·Level 변수들은 서로 높은 상관을 보일 수 있어 다중공선성이 발생할 수 있습니다.
> Ridge/Lasso/ElasticNet 규제 또는 Lasso 변수 선택으로 완화합니다.

---## 3) `date` 컬럼 시계열 분석측정 시각(`date`)이 실리카 농도 예측에 유의미한지 확인합니다.- 시계열 추이·자기상관(ACF) 분석- date 파생변수만으로 예측 가능한지- 공정 변수 + date 파생변수 모델과 공정 변수만 사용 모델 성능 비교

In [ ]:
ts = df.set_index("date")[TARGET].sort_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(ts.index, ts.values, linewidth=0.6, color="#4C72B0")
axes[0].set_title("Silica Concentrate — Time Series")
axes[0].set_ylabel(TARGET)

# 7일 이동평균
rolling_mean = ts.rolling(window=24 * 7).mean()
axes[1].plot(ts.index, rolling_mean, color="#DD8452")
axes[1].set_title("7-day Rolling Mean")
axes[1].set_ylabel("Rolling Mean")
plt.tight_layout()
plt.show()

print("기간:", ts.index.min(), "~", ts.index.max())
print("관측 수:", len(ts))

In [ ]:
# 정상성 검정 (ADF)
adf_stat, adf_p, *_ = adfuller(ts.dropna())
print(f"ADF statistic: {adf_stat:.4f}")
print(f"ADF p-value  : {adf_p:.2e}")
print("→ p < 0.05 이면 정상(stationary) 시계열로 해석")

fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(ts.dropna(), lags=48, ax=ax)
ax.set_title("ACF — % Silica Concentrate")
plt.tight_layout()
plt.show()

acf_vals = acf(ts.dropna(), nlags=48, fft=True)
print(f"ACF lag-1: {acf_vals[1]:.3f}  |  ACF lag-24: {acf_vals[24]:.3f}")

In [ ]:
# date 파생변수 생성
df_ts = df.copy()
df_ts["days_since_start"] = (df_ts["date"] - df_ts["date"].min()).dt.total_seconds() / 86400
df_ts["hour"] = df_ts["date"].dt.hour
df_ts["dayofweek"] = df_ts["date"].dt.dayofweek

date_features = ["days_since_start", "hour", "dayofweek"]
process_features = [c for c in df.columns if c not in EXCLUDE_COLS + [TARGET]]

Y_all = df_ts[TARGET]

def quick_eval(X, label):
    X_train, X_test, y_train, y_test = train_test_split(
        X, Y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    model = LinearRegression().fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    cv_rmse = -cross_val_score(
        LinearRegression(), X, Y_all, cv=5,
        scoring="neg_root_mean_squared_error",
    ).mean()
    return {"Scenario": label, "RMSE": rmse, "R2": r2, "CV_RMSE": cv_rmse}

date_eval = pd.DataFrame([
    quick_eval(df_ts[date_features], "date 파생변수만"),
    quick_eval(df_ts[process_features], "공정 변수만 (date 제외)"),
    quick_eval(df_ts[process_features + date_features], "공정 변수 + date 파생변수"),
]).round(4)

date_eval

In [ ]:
# date 파생변수와 타겟 상관계수
df_ts[date_features + [TARGET]].corr()[TARGET].drop(TARGET)

### 시계열 분석 결론| 관찰 | 해석 ||:---|:---|| 타겟 시계열 ACF lag-1이 높음 | 시간 순서상 인접 시점의 실리카 농도가 유사함 || date 파생변수 단독 R² ≈ 0 | 달력·시각 정보만으로는 실리카를 거의 설명 못함 || 공정 변수 + date 파생변수 | 테스트 RMSE는 소폭 개선되나 **CV RMSE는 오히려 악화** |**결정: `date` 컬럼 및 파생변수는 회귀 모델에서 제외합니다.**공정 변수(원료·시약·플로테이션 조건)가 실리카 농도를 설명하는 핵심 정보이며, 측정 시각 자체는 예측력이 미미합니다.

---
## 4) 입출력 분할 및 학습/테스트 분할

In [ ]:
Y = df[TARGET]
X = df.drop(EXCLUDE_COLS + [TARGET], axis=1)

print("입력변수:", X.shape)
print("출력변수:", Y.shape)
X.columns.tolist()

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("학습:", X_train.shape, Y_train.shape)
print("테스트:", X_test.shape, Y_test.shape)

In [ ]:
# 회귀 성능 출력 함수 (실습 노트북과 동일)
def get_regscore(true, pred):
    print("MSE       : %.3f" % mean_squared_error(true, pred))
    print("RMSE      : %.3f" % np.sqrt(mean_squared_error(true, pred)))
    print("MAE       : %.3f" % mean_absolute_error(true, pred))
    print("R-squared : %.3f" % r2_score(true, pred))

---## 5) 다중 선형 회귀 (OLS)

In [ ]:
LR_model = LinearRegression()
LR_model.fit(X_train, Y_train)
Y_pred_LR = LR_model.predict(X_test)
get_regscore(Y_test, Y_pred_LR)

In [ ]:
Coef_LR = pd.DataFrame({"Variable": X_train.columns, "Coefficient": LR_model.coef_})
Coef_LR.sort_values("Coefficient", key=abs, ascending=False).head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(Y_train, LR_model.predict(X_train), alpha=0.4)
axes[0].plot([Y_train.min(), Y_train.max()], [Y_train.min(), Y_train.max()], "r--")
axes[0].set_xlabel("Real y")
axes[0].set_ylabel("Predicted y")
axes[0].set_title("Train")

axes[1].scatter(Y_test, Y_pred_LR, alpha=0.4)
axes[1].plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()], "r--")
axes[1].set_xlabel("Real y")
axes[1].set_ylabel("Predicted y")
axes[1].set_title("Test")

plt.tight_layout()
plt.show()

---## 6) Ridge 회귀 — alpha 최적화실습 노트북에서는 alpha=0.1, 10을 비교했습니다. 여기서는 `GridSearchCV`로 최적 alpha를 탐색합니다.

In [ ]:
alpha_range = np.logspace(-3, 3, 50)
rmse_scorer = make_scorer(
    lambda y_true, y_pred: -np.sqrt(mean_squared_error(y_true, y_pred)),
    greater_is_better=True,
)

ridge_grid = GridSearchCV(
    Ridge(),
    param_grid={"alpha": alpha_range},
    scoring=rmse_scorer,
    cv=5,
)
ridge_grid.fit(X_train, Y_train)

best_ridge_alpha = ridge_grid.best_params_["alpha"]
Ridge_model = ridge_grid.best_estimator_
Y_pred_Ridge = Ridge_model.predict(X_test)

print(f"최적 Ridge alpha: {best_ridge_alpha:.4f}")
get_regscore(Y_test, Y_pred_Ridge)

In [ ]:
# alpha에 따른 CV RMSE 변화
cv_results = pd.DataFrame(ridge_grid.cv_results_)
plt.figure(figsize=(8, 4))
plt.semilogx(cv_results["param_alpha"], -cv_results["mean_test_score"])
plt.axvline(best_ridge_alpha, color="red", linestyle="--", label=f"best={best_ridge_alpha:.4f}")
plt.xlabel("alpha")
plt.ylabel("CV RMSE")
plt.title("Ridge — CV RMSE vs alpha")
plt.legend()
plt.tight_layout()
plt.show()

---## 7) Lasso 회귀 — alpha 최적화

In [ ]:
lasso_grid = GridSearchCV(
    Lasso(max_iter=10000),
    param_grid={"alpha": alpha_range},
    scoring=rmse_scorer,
    cv=5,
)
lasso_grid.fit(X_train, Y_train)

best_lasso_alpha = lasso_grid.best_params_["alpha"]
Lasso_model = lasso_grid.best_estimator_
Y_pred_Lasso = Lasso_model.predict(X_test)

print(f"최적 Lasso alpha: {best_lasso_alpha:.4f}")
get_regscore(Y_test, Y_pred_Lasso)

In [ ]:
Coef_Lasso = pd.DataFrame({"Variable": X_train.columns, "Coefficient": Lasso_model.coef_})
print("Lasso가 0으로 만든 변수:")
display(Coef_Lasso.loc[Coef_Lasso["Coefficient"] == 0])
print("선택된 변수:")
display(Coef_Lasso.loc[Coef_Lasso["Coefficient"] != 0].sort_values("Coefficient", key=abs, ascending=False))

---## 8) ElasticNet — Ridge(L2) + Lasso(L1) 결합L1·L2 규제를 동시에 적용해 다중공선성 완화와 변수 선택을 함께 수행합니다.

In [ ]:
enet_grid = GridSearchCV(
    ElasticNet(max_iter=10000),
    param_grid={
        "alpha": np.logspace(-3, 1, 30),
        "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
    },
    scoring=rmse_scorer,
    cv=5,
)
enet_grid.fit(X_train, Y_train)

ElasticNet_model = enet_grid.best_estimator_
Y_pred_EN = ElasticNet_model.predict(X_test)

print("최적 ElasticNet:", enet_grid.best_params_)
get_regscore(Y_test, Y_pred_EN)

---## 9) 모델 비교

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        f"Ridge (alpha={best_ridge_alpha:.4f})",
        f"Lasso (alpha={best_lasso_alpha:.4f})",
        f"ElasticNet (alpha={enet_grid.best_params_['alpha']:.4f}, l1={enet_grid.best_params_['l1_ratio']})",
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(Y_test, Y_pred_LR)),
        np.sqrt(mean_squared_error(Y_test, Y_pred_Ridge)),
        np.sqrt(mean_squared_error(Y_test, Y_pred_Lasso)),
        np.sqrt(mean_squared_error(Y_test, Y_pred_EN)),
    ],
    "MAE": [
        mean_absolute_error(Y_test, Y_pred_LR),
        mean_absolute_error(Y_test, Y_pred_Ridge),
        mean_absolute_error(Y_test, Y_pred_Lasso),
        mean_absolute_error(Y_test, Y_pred_EN),
    ],
    "R-squared": [
        r2_score(Y_test, Y_pred_LR),
        r2_score(Y_test, Y_pred_Ridge),
        r2_score(Y_test, Y_pred_Lasso),
        r2_score(Y_test, Y_pred_EN),
    ],
}).round(3)

results.sort_values("RMSE")

In [ ]:
plt.figure(figsize=(15, 6))
x_pos = np.arange(len(X_train.columns))
width = 0.2

plt.bar(x_pos - 1.5 * width, LR_model.coef_, width, label="Linear Regression", alpha=0.85)
plt.bar(x_pos - 0.5 * width, Ridge_model.coef_, width, label="Ridge", alpha=0.85)
plt.bar(x_pos + 0.5 * width, Lasso_model.coef_, width, label="Lasso", alpha=0.85)
plt.bar(x_pos + 1.5 * width, ElasticNet_model.coef_, width, label="ElasticNet", alpha=0.85)

plt.xticks(x_pos, X_train.columns, rotation=60, ha="right")
plt.ylabel("Coefficient")
plt.title("Regression Coefficients Comparison")
plt.legend()
plt.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Lasso 선택 변수 기준 상관행렬 비교
selected_vars = Coef_Lasso.loc[Coef_Lasso["Coefficient"] != 0, "Variable"].tolist()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(X_train.corr(), vmin=-1, vmax=1, center=0, cmap="coolwarm", ax=axes[0], square=True)
axes[0].set_title("All Variables")

if selected_vars:
    sns.heatmap(X_train[selected_vars].corr(), vmin=-1, vmax=1, center=0, cmap="coolwarm", ax=axes[1], square=True, annot=True, fmt=".2f")
    axes[1].set_title("Lasso Selected Variables")
else:
    axes[1].set_title("Lasso Selected Variables (none)")

plt.tight_layout()
plt.show()

---
## 10) 최종 정리

1. **타겟**: `% Silica Concentrate` (정광 실리카 불순물 비율)
2. **`date` 처리**: 시계열 자기상관은 존재하나, 달력 파생변수는 예측 기여가 미미하고 CV 성능도 개선되지 않아 **제외**
3. **다중공선성**: Flotation 컬럼 변수 간 상관이 높아 Ridge/Lasso/ElasticNet이 OLS 대비 안정적
4. **최적 모델**: 위 성능 비교표에서 RMSE가 가장 낮은 모델을 최종 선택